# Corporate Innovation: Predictive Modeling and Interpretation

**Research question:** Which firm and stakeholder characteristics are most informative for predicting next-year R&D investment and invention patent output?

The project integrates 75 predictors across 11 categories. The final thesis reports 13,611 complete firm-year observations from the CSMAR/CNRDS source sample covering 2012-2024. This notebook organizes the existing model-comparison and SHAP code into one workflow, removing duplicate alternative implementations and manual-input plot blocks.

## Contents
0. Environment and configuration
1. Data preparation and feature framework
2. Data understanding and sample checks
3. Time-based train/test design
4. Model training and selection
5. Predictive performance
6. SHAP feature and group importance
7. Dependence patterns
8. Temporal analysis
9. Final-window interaction analysis
10. Research summary


## Part 0: Environment and configuration

Use the authorized prepared Stata dataset in `data/MERGED_DATA_Winsor.dta`. Full grid search and SHAP calculations are computationally expensive and can be enabled independently below. Select either target without changing the feature list or time splits.


In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import shap
import statsmodels.api as sm
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

DATA_PATH = DATA_DIR / "MERGED_DATA_Winsor.dta"
target_col = "RDCost_t1"  # Alternative target: "InvPatent_t1".
TARGET_LABELS = {"RDCost_t1": "R&D investment intensity", "InvPatent_t1": "Invention patent output"}
assert target_col in TARGET_LABELS
target_label = TARGET_LABELS[target_col]
RUN_MODEL_EVALUATION = False
RUN_SHAP = False
RUN_INTERACTIONS = False
plt.rcParams["font.family"] = "DejaVu Serif"

from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")


## Part 1: Data preparation and feature framework

Firm-level merging, variable construction, sample selection and winsorization precede this notebook in the accompanying [Stata scripts](../stata/). The prepared dataset is loaded here; it is not reconstructed from raw exports or winsorized a second time.

The targets are next-year R&D expenditure divided by sales (`RDCost_t1`) and log(next-year invention patent applications + 1) (`InvPatent_t1`). The feature categories cover firm fundamentals and internal and external stakeholders.


In [2]:
feature_cols = ['TotalAssets', 'Age', 'DebtRatio', 'Cashflow', 'Growth', 'ProfitMargin', 'ROA', 'FixedAssetRatio', 'EV', 'OverseasInv', 'OverseasRev', 'MA', 'ESG', 'CEORisk', 'CEODegree', 'IsMBA', 'FinBack', 'CEOAcademicBack', 'Versatility', 'Top3ManageSumSalary', 'CEOIncentive', 'REM', 'IntControl', 'KV', 'Deficiency', 'EmployeeTurnover', 'ESOP', 'Paygap', 'SocInsurance', 'EmployeeSalary', 'InvestorShare', 'Ownstructure', 'MajorShareholders', 'Ownership', 'Litigation', 'VAM', 'Boardsize', 'IndDirectorRatio', 'FemaleDirector', 'DO', 'DCG', 'AISkill', 'DataSkill', 'ChairAcademicBack', 'FC', 'BOND_D', 'BOND_P', 'BankHHI', 'Top3Bank', 'BankCR', 'AnaAttention', 'FinNews', 'NegFinNews', 'FERR', 'FOPT', 'Financeavailable', 'Shortavailable', 'Liquidity', 'NCSKEW', 'DUVOL', 'SC', 'GovSub', 'Dige', 'Gdatact', 'Ipp', 'HTZone', 'EnvSubsidy', 'Enrelcost', 'TaxRate', 'Talent', 'IP_EncImp', 'HHI', 'Zombie_Ratio', 'SupplierHHI', 'CustomerHHI']

raw_groups = {'Fundamental': ['TotalAssets', 'Age', 'DebtRatio', 'Cashflow', 'Growth', 'ProfitMargin', 'ROA', 'FixedAssetRatio', 'EV', 'OverseasInv', 'OverseasRev', 'MA', 'ESG'], 'Industry': ['HHI', 'Zombie_Ratio'], 'Manager': ['CEORisk', 'CEODegree', 'IsMBA', 'FinBack', 'CEOAcademicBack', 'Versatility', 'Top3ManageSumSalary', 'CEOIncentive', 'REM', 'IntControl', 'KV', 'Deficiency'], 'Employee': ['EmployeeTurnover', 'ESOP', 'Paygap', 'SocInsurance', 'EmployeeSalary'], 'Shareholder': ['InvestorShare', 'Ownstructure', 'MajorShareholders', 'Ownership', 'Litigation', 'VAM'], 'Board': ['Boardsize', 'IndDirectorRatio', 'FemaleDirector', 'DO', 'DCG', 'AISkill', 'DataSkill', 'ChairAcademicBack'], 'Creditor': ['FC', 'BOND_D', 'BOND_P', 'BankHHI', 'Top3Bank', 'BankCR'], 'Government': ['GovSub', 'Dige', 'Gdatact', 'Ipp', 'HTZone', 'EnvSubsidy', 'Enrelcost', 'TaxRate', 'Talent', 'IP_EncImp'], 'Infomediary': ['AnaAttention', 'FinNews', 'NegFinNews', 'FERR', 'FOPT'], 'Stock Market': ['Financeavailable', 'Shortavailable', 'Liquidity', 'NCSKEW', 'DUVOL', 'SC'], 'Supply Chain': ['SupplierHHI', 'CustomerHHI']}

assert len(feature_cols) == 75
assert len(raw_groups) == 11
assert set(feature_cols) == {feature for group in raw_groups.values() for feature in group}


In [3]:
if not DATA_PATH.exists():
    raise FileNotFoundError("Place the authorized MERGED_DATA_Winsor.dta dataset in the data folder.")
raw_data = pd.read_stata(DATA_PATH)
required_columns = set(feature_cols + ["year", "stock", "complete_obs", target_col])
assert required_columns.issubset(raw_data.columns), "Required research variables are missing."
data = raw_data.loc[raw_data["complete_obs"].eq(0)].copy()
print(f"Source rows: {len(raw_data):,}; complete-case rows: {len(data):,}")


Source rows: 69,865; complete-case rows: 13,611


## Part 2: Data understanding and sample checks

Inspect observation counts, coverage and variable distributions without displaying proprietary firm-level records. The complete-case flag follows the existing preparation scripts. The per-window function below retains the source treatment of non-finite predictors and fits its scaler on training observations only.


In [4]:
sample_summary = pd.Series({"firm_year_observations": len(data), "firms": data["stock"].nunique(), "first_feature_year": data["year"].min(), "last_feature_year": data["year"].max(), "predictors": len(feature_cols), "categories": len(raw_groups)})
sample_summary


firm_year_observations    13611.0
firms                      2617.0
first_feature_year         2012.0
last_feature_year          2023.0
predictors                   75.0
categories                   11.0
dtype: float64

In [5]:
quality_checks = pd.Series({"missing_predictors": int(data[feature_cols].isna().sum().sum()), "missing_target": int(data[target_col].isna().sum()), "duplicate_firm_year_rows": int(data.duplicated(["stock", "year"]).sum())})
quality_checks


missing_predictors          33
missing_target               0
duplicate_firm_year_rows     0
dtype: int64

In [6]:
data[[target_col]].describe()


          RDCost_t1
count  13611.000000
mean       0.060828
std        0.060971
min        0.000254
25%        0.026952
50%        0.043509
75%        0.072402
max        0.367364

## Part 3: Time-based train/test design

The first training period is 2012-2014 and the first test feature year is 2016. The training period expands by one year for each later test. The gap separates next-year training labels from test-period features. This is an expanding-window evaluation rather than a random train/test split.


In [7]:
train_test_splits = [([2012, 2013, 2014], 2016), ([2012, 2013, 2014, 2015], 2017), ([2012, 2013, 2014, 2015, 2016], 2018), ([2012, 2013, 2014, 2015, 2016, 2017], 2019), ([2012, 2013, 2014, 2015, 2016, 2017, 2018], 2020), ([2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019], 2021), ([2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020], 2022), ([2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021], 2023)]

pd.DataFrame([{"training_start": min(years), "training_end": max(years), "test_year": test_year} for years, test_year in train_test_splits])


   training_start  training_end  test_year
0            2012          2014       2016
1            2012          2015       2017
2            2012          2016       2018
3            2012          2017       2019
4            2012          2018       2020
5            2012          2019       2021
6            2012          2020       2022
7            2012          2021       2023

In [8]:
def prepare_year_data(data, train_years, test_year, target_col='RDCost_t1'):
    complete_data = data[data['complete_obs'] == 0].copy()
    train_data = complete_data[complete_data['year'].isin(train_years)].copy()
    test_data = complete_data[complete_data['year'] == test_year].copy()
    X_train = train_data[feature_cols].copy()
    y_train = train_data[target_col].copy()
    X_test = test_data[feature_cols].copy()
    y_test = test_data[target_col].copy()
    for col in feature_cols:
        X_train[col] = X_train[col].replace([np.inf, -np.inf], np.nan)
        X_test[col] = X_test[col].replace([np.inf, -np.inf], np.nan)
    train_mask = y_train.notna() & X_train.notna().all(axis=1)
    X_train_clean = X_train[train_mask]
    y_train_clean = y_train[train_mask]
    test_mask = y_test.notna() & X_test.notna().all(axis=1)
    X_test_clean = X_test[test_mask]
    y_test_clean = y_test[test_mask]
    print(f'Training set: {len(X_train_clean)} observations (years: {train_years})')
    print(f'Test set: {len(X_test_clean)} observations (years: {test_year})')
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_clean)
    X_test_scaled = scaler.transform(X_test_clean)
    return (X_train_scaled, y_train_clean.values, X_test_scaled, y_test_clean.values, X_test_clean, y_test_clean)


## Part 4: Model training and selection

Compare GBRT (named `GBDT` in the source code), Random Forest and an OLS baseline. Retain the original parameter grids, random seeds and within-training cross-validation settings: five folds for GBRT, three for RF, with no parameter search for OLS. These internal folds are distinct from the expanding-window held-out evaluation.


In [9]:
def time_series_evaluation(X_train, y_train, X_test, y_test, model_name='GBDT'):
    if len(X_train) < 10 or len(X_test) < 5:
        print('Insufficient observations; skipping this split.')
        return None
    try:
        cv_folds = 5
        if model_name == 'GBDT':
            model = GradientBoostingRegressor(random_state=42)
            param_grid = {'learning_rate': [0.1, 0.2], 'n_estimators': [100, 150], 'max_depth': [3, 5]}
        elif model_name == 'RF':
            model = RandomForestRegressor(random_state=42, n_jobs=-1)
            param_grid = {'n_estimators': [100, 150], 'max_depth': [5, 8, 10], 'max_features': ['sqrt', 1.0]}
            cv_folds = 3
        elif model_name == 'OLS':
            ols_model = LinearRegression()
            ols_model.fit(X_train, y_train)
            y_pred = ols_model.predict(X_test)
            r2 = r2_score(y_test, y_pred)
            mse = mean_squared_error(y_test, y_pred)
            rmse = np.sqrt(mse)
            mae = np.mean(np.abs(y_test - y_pred))
            results = {'r2': r2, 'mse': mse, 'rmse': rmse, 'mae': mae, 'n_train': len(X_train), 'n_test': len(X_test), 'model': ols_model, 'best_params': 'OLS (no hyperparameters)'}
            return results
        print(f'Running {model_name} grid search...')
        grid_search = GridSearchCV(model, param_grid, cv=cv_folds, scoring='r2', n_jobs=-1, verbose=0)
        grid_search.fit(X_train, y_train)
        print(f'Best parameters: {grid_search.best_params_}')
        best_model = grid_search.best_estimator_
        y_pred = best_model.predict(X_test)
        r2 = r2_score(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        mae = np.mean(np.abs(y_test - y_pred))
        results = {'r2': r2, 'mse': mse, 'rmse': rmse, 'mae': mae, 'n_train': len(X_train), 'n_test': len(X_test), 'model': best_model, 'best_params': grid_search.best_params_}
        return results
    except Exception as e:
        print(f'Model fitting failed: {e}')
        return None


In [10]:
all_results = {}
if RUN_MODEL_EVALUATION:
    for model_name in ["GBDT", "RF", "OLS"]:
        detailed_results = []
        yearly_results = {}
        for train_years, test_year in train_test_splits:
            X_train, y_train, X_test, y_test, _, _ = prepare_year_data(data, train_years, test_year, target_col)
            result = time_series_evaluation(X_train, y_train, X_test, y_test, model_name)
            if result is not None:
                detailed_results.append(result)
                yearly_results[f"{min(train_years)}-{max(train_years)}_to_{test_year}"] = result
        all_results[model_name] = {"yearly_results": yearly_results, "detailed_results": detailed_results}
else:
    print("Set RUN_MODEL_EVALUATION=True to rerun the full model-comparison grid searches.")


Set RUN_MODEL_EVALUATION=True to rerun the full model-comparison grid searches.


## Part 5: Predictive performance

The functions below summarize computed results when model evaluation is enabled. The reference table is transcribed from the final thesis, Table 4-2; it is not presented as output from a new run of this consolidated notebook.

| Target | GBRT R² | RF R² | OLS R² |
| --- | ---: | ---: | ---: |
| R&D investment intensity | 0.5428 | 0.5227 | 0.4761 |
| Invention patent output | 0.4478 | 0.4338 | 0.3541 |

Table 4-2 also reports MSE, RMSE and MAE. GBRT has the lowest errors and highest R² for both targets.


In [11]:
def plot_r2_boxplot(all_results):
    r2_data = []
    for model_name in ['GBDT', 'RF', 'OLS']:
        if model_name in all_results and all_results[model_name]['detailed_results']:
            detailed_results = all_results[model_name]['detailed_results']
            r2_values = [result['r2'] for result in detailed_results]
            for r2 in r2_values:
                r2_data.append({'Model': model_name, 'R²': r2})
    if not r2_data:
        print('Insufficient data for the boxplot.')
        return
    r2_df = pd.DataFrame(r2_data)
    plt.figure(figsize=(10, 6))
    colors = ['red', 'orange', 'yellow']
    box_plot = sns.boxplot(data=r2_df, x='Model', y='R²', palette=colors, showfliers=False)
    means = []
    for model in ['GBDT', 'RF', 'OLS']:
        model_data = r2_df[r2_df['Model'] == model]['R²']
        if len(model_data) > 0:
            mean_val = model_data.mean()
            means.append(mean_val)
        else:
            means.append(None)
    for i, mean_val in enumerate(means):
        if mean_val is not None:
            plt.plot(i, mean_val, 'o', markersize=6, markeredgewidth=1, markeredgecolor='black', markerfacecolor='white')
    Q1 = r2_df['R²'].quantile(0.25)
    Q3 = r2_df['R²'].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    visible_data = r2_df[(r2_df['R²'] >= lower_bound) & (r2_df['R²'] <= upper_bound)]
    y_min_visible = visible_data['R²'].min()
    y_max_visible = visible_data['R²'].max()
    padding = 0.05
    plt.ylim(y_min_visible - padding, y_max_visible + padding)
    ax = plt.gca()
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.1))
    plt.xlabel('Model', fontsize=12)
    plt.ylabel('R²', fontsize=12)
    plt.grid(axis='y', alpha=0.3)
    plt.title('Time-Series Dispersion of R² - R&D Cost', fontsize=14, pad=15)
    plt.tight_layout()
    plt.show()


In [12]:
def print_performance_table(all_results):
    metrics = ['MSE', 'RMSE', 'MAE', 'R²']
    models = ['GBDT', 'RF', 'OLS']
    performance_data = []
    for model_name in models:
        if model_name in all_results and all_results[model_name]['detailed_results']:
            detailed_results = all_results[model_name]['detailed_results']
            avg_mse = np.mean([result['mse'] for result in detailed_results])
            avg_rmse = np.mean([result['rmse'] for result in detailed_results])
            avg_mae = np.mean([result['mae'] for result in detailed_results])
            avg_r2 = np.mean([result['r2'] for result in detailed_results])
            performance_data.append({'Model': model_name, 'MSE': avg_mse, 'RMSE': avg_rmse, 'MAE': avg_mae, 'R²': avg_r2})
    if not performance_data:
        print('No performance results available.')
        return
    performance_df = pd.DataFrame(performance_data)
    performance_df.set_index('Model', inplace=True)
    print('\n' + '=' * 80)
    print('Model performance comparison')
    print('=' * 80)
    print(f"{'Metric':<10} {'GBDT':<12} {'RF':<12} {'OLS':<12}")
    print('-' * 50)
    for metric in metrics:
        gbdt_val = performance_df.loc['GBDT', metric] if 'GBDT' in performance_df.index else 'N/A'
        rf_val = performance_df.loc['RF', metric] if 'RF' in performance_df.index else 'N/A'
        ols_val = performance_df.loc['OLS', metric] if 'OLS' in performance_df.index else 'N/A'
        if metric != 'R²':
            print(f'{metric:<10} {gbdt_val:<12.4f} {rf_val:<12.4f} {ols_val:<12.4f}')
        else:
            print(f'{metric:<10} {gbdt_val:<12.4f} {rf_val:<12.4f} {ols_val:<12.4f}')


In [13]:
if RUN_MODEL_EVALUATION:
    plot_r2_boxplot(all_results)
    print_performance_table(all_results)


## Part 6: SHAP feature and group importance

Fit GBRT within each expanding window using the source SHAP workflow's three-fold grid search, then explain the held-out observations for that window. Pool those out-of-sample SHAP values to construct relative importance scores. This separate interpretation fit retains the source configuration and is not silently substituted for the comparison fits in Part 4.


In [14]:
if RUN_SHAP:
    all_test_X = []
    all_test_shap_values = []
    r2_scores = []
    gbdt_param_grid = {'learning_rate': [0.1, 0.2], 'n_estimators': [100, 150], 'max_depth': [3, 5]}
    print('\nStarting expanding-window training and SHAP calculation...')
    if 'year' not in data.columns:
        raise ValueError('The dataset must contain a year column for time splits.')
    for train_years, test_year in train_test_splits:
        print(f'\nWindow: training years {train_years} -> test year {test_year}')
        X_train = data.loc[data['year'].isin(train_years), feature_cols]
        y_train = data.loc[data['year'].isin(train_years), target_col]
        X_test = data.loc[data['year'] == test_year, feature_cols]
        y_test = data.loc[data['year'] == test_year, target_col]
        if len(X_test) == 0:
            print(f'Notice: {test_year} has no test observations; skipping.')
            continue
        gbdt = GradientBoostingRegressor(random_state=42)
        grid_search = GridSearchCV(gbdt, gbdt_param_grid, cv=3, scoring='r2', n_jobs=-1)
        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_
        y_pred = best_model.predict(X_test)
        score = r2_score(y_test, y_pred)
        r2_scores.append(score)
        print(f'   Best parameters: {grid_search.best_params_}')
        print(f'   Test R2: {score:.4f}')
        explainer = shap.TreeExplainer(best_model)
        shap_values_test = explainer.shap_values(X_test)
        all_test_X.append(X_test)
        all_test_shap_values.append(shap_values_test)
    print(f'\nMean test R2 across completed windows: {np.mean(r2_scores):.4f}')
    if not all_test_X:
        raise ValueError('No test observations were collected.')
    combined_X = pd.concat(all_test_X, axis=0)
    combined_shap_values = np.vstack(all_test_shap_values)
    print(f'Pooled test observations (2016-2023): {len(combined_X)}')
else:
    print("Set RUN_SHAP=True to rerun the window-specific SHAP workflow.")


Set RUN_SHAP=True to rerun the window-specific SHAP workflow.


### Individual-feature relative importance

Normalize mean absolute SHAP values and aggregate the fundamental variables into one baseline category for the individual-feature ranking.


In [15]:
if RUN_SHAP:
    print('\nGenerating the relative importance table...')
    raw_importance = np.abs(combined_shap_values).mean(axis=0)
    feature_names = combined_X.columns.tolist()
    import_dict = dict(zip(feature_names, raw_importance))
    fundamental_features_list = ['TotalAssets', 'Age', 'DebtRatio', 'Cashflow', 'Growth', 'ProfitMargin', 'ROA', 'FixedAssetRatio', 'EV', 'OverseasInv', 'OverseasRev', 'MA', 'ESG']
    fundamental_score = 0
    final_features_dict = {}
    for feat, score in import_dict.items():
        if feat in fundamental_features_list:
            fundamental_score += score
        else:
            final_features_dict[feat] = score
    final_features_dict['Fundamental_Features'] = fundamental_score
    total_score = sum(final_features_dict.values())
    rvi_dict = {k: v / total_score * 100 for k, v in final_features_dict.items()}
    sorted_rvi = sorted(rvi_dict.items(), key=lambda x: x[1], reverse=True)
    top_10_rvi = sorted_rvi[:10]
    print('-' * 50)
    print(f"{'Rank':<5} | {'Variable':<25} | {'RVI Scores(%)':<15}")
    print('-' * 50)
    for rank, (var, score) in enumerate(top_10_rvi, 1):
        print(f'{rank:<5} | {var:<25} | {score:.2f}')
    print('-' * 50)


### Feature-group relative importance

Aggregate feature contributions by stakeholder category, identify each group's leading feature and inspect variation across test years.


In [16]:
if RUN_SHAP:
    raw_groups = {'Fundamental': ['TotalAssets', 'Age', 'DebtRatio', 'Cashflow', 'Growth', 'ProfitMargin', 'ROA', 'FixedAssetRatio', 'EV', 'OverseasInv', 'OverseasRev', 'MA', 'ESG'], 'Industry': ['HHI', 'Zombie_Ratio'], 'Manager': ['CEORisk', 'CEODegree', 'IsMBA', 'FinBack', 'CEOAcademicBack', 'Versatility', 'Top3ManageSumSalary', 'CEOIncentive', 'REM', 'IntControl', 'KV', 'Deficiency'], 'Employee': ['EmployeeTurnover', 'ESOP', 'Paygap', 'SocInsurance', 'EmployeeSalary'], 'Shareholder': ['InvestorShare', 'Ownstructure', 'MajorShareholders', 'Ownership', 'Litigation', 'VAM'], 'Board': ['Boardsize', 'IndDirectorRatio', 'FemaleDirector', 'DO', 'DCG', 'AISkill', 'DataSkill', 'ChairAcademicBack'], 'Creditor': ['FC', 'BOND_D', 'BOND_P', 'BankHHI', 'Top3Bank', 'BankCR'], 'Government': ['GovSub', 'Dige', 'Gdatact', 'Ipp', 'HTZone', 'EnvSubsidy', 'Enrelcost', 'TaxRate', 'Talent', 'IP_EncImp'], 'Infomediary': ['AnaAttention', 'FinNews', 'NegFinNews', 'FERR', 'FOPT'], 'Stock Market': ['Financeavailable', 'Shortavailable', 'Liquidity', 'NCSKEW', 'DUVOL', 'SC'], 'Supply Chain': ['SupplierHHI', 'CustomerHHI']}
    feature_groups = {}
    valid_features = set(feature_cols)
    for grp_name, feat_list in raw_groups.items():
        valid_list = [f for f in feat_list if f in valid_features]
        if valid_list:
            feature_groups[grp_name] = valid_list
    global_mean_abs_shap = np.abs(combined_shap_values).mean(axis=0)
    global_import_dict = dict(zip(feature_cols, global_mean_abs_shap))
    total_global_importance = sum(global_import_dict.values())
    group_data = []
    for grp, feats in feature_groups.items():
        grp_score_raw = sum((global_import_dict[f] for f in feats))
        grp_rvi = grp_score_raw / total_global_importance
        max_feat = max(feats, key=lambda f: global_import_dict[f])
        max_feat_score_raw = global_import_dict[max_feat]
        max_feat_rvi = max_feat_score_raw / total_global_importance
        group_data.append({'Group': grp, 'Group_RVI': grp_rvi, 'Top_Feature': max_feat, 'Top_Feature_RVI': max_feat_rvi})
    df_group = pd.DataFrame(group_data).sort_values('Group_RVI', ascending=False)
    yearly_points = {grp: [] for grp in df_group['Group']}
    for i, shap_matrix in enumerate(all_test_shap_values):
        year_mean_abs = np.abs(shap_matrix).mean(axis=0)
        year_dict = dict(zip(feature_cols, year_mean_abs))
        year_total = sum(year_dict.values())
        for grp in df_group['Group']:
            feats = feature_groups[grp]
            grp_score = sum((year_dict[f] for f in feats))
            yearly_points[grp].append(grp_score / year_total)
    import matplotlib.cm as cm
    from matplotlib.patches import Patch
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(20, 8))
    colors = cm.rainbow(np.linspace(1, 0, len(df_group)))
    x_pos = np.arange(len(df_group))
    bar_width = 0.35
    bars_group = ax.bar(x_pos - bar_width / 2, df_group['Group_RVI'], width=bar_width, color=colors, alpha=0.85, label='Group Importance (Total)', zorder=1)
    bars_top = ax.bar(x_pos + bar_width / 2, df_group['Top_Feature_RVI'], width=bar_width, color=colors, alpha=0.4, label='Top Feature (Individual)', zorder=1)
    for idx, row in enumerate(df_group.itertuples()):
        grp_name = row.Group
        points = yearly_points[grp_name]
        x_scatter = [x_pos[idx] - bar_width / 2] * len(points)
        ax.scatter(x_scatter, points, color=colors[idx], s=35, alpha=0.6, zorder=2, edgecolor='none')
        ax.text(x_pos[idx], row.Top_Feature_RVI + 0.002, row.Top_Feature, ha='left', va='bottom', fontsize=8, rotation=0, color='black')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(df_group['Group'], rotation=45, ha='right', fontsize=12)
    ax.set_ylabel('Normalized SHAP Importance (RVI)', fontsize=12)
    ax.set_title('Feature Group Importance with Top Feature - R&D Cost', fontsize=16, fontweight='bold')
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    legend_elements = [Patch(facecolor='red', alpha=0.85, label='Group Total Importance'), Patch(facecolor='red', alpha=0.4, label='Top Feature in Group'), plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=8, label='Yearly RVI (Scatter)')]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=10)
    plt.tight_layout()
    plt.show()


## Part 7: Dependence patterns

Inspect the top individual features and selected-feature panels. The smoothed lines summarize the fitted models' prediction contributions; they do not identify causal response curves.


In [17]:
if RUN_SHAP:
    print('\nPlotting dependence patterns for the top 20 features...')
    mean_abs_shap_raw = np.abs(combined_shap_values).mean(axis=0)
    top_indices_raw = np.argsort(mean_abs_shap_raw)[::-1][:20]
    top_features_raw = [combined_X.columns[i] for i in top_indices_raw]
    for feature_name in top_features_raw:
        fig, ax = plt.subplots(figsize=(8, 6))
        shap.dependence_plot(feature_name, combined_shap_values, combined_X, interaction_index=None, ax=ax, show=False, alpha=0.2, dot_size=15)
        x_data = combined_X[feature_name].values
        y_data = combined_shap_values[:, combined_X.columns.get_loc(feature_name)]
        sort_idx = np.argsort(x_data)
        x_sorted = x_data[sort_idx]
        y_sorted = y_data[sort_idx]
        window_size = int(len(x_sorted) * 0.05)
        if window_size < 1:
            window_size = 1
        y_rolling = pd.Series(y_sorted).rolling(window=window_size, center=True).mean()
        ax.plot(x_sorted, y_rolling, color='black', linewidth=2.5, label='Mean Effect')
        ax.axhline(0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
        ax.set_title(f'{feature_name} -> {target_label}', fontsize=12)
        ax.set_xlabel(f'Feature Value: {feature_name}', fontsize=10)
        ax.set_ylabel('SHAP Value', fontsize=10)
        ax.legend()
        plt.tight_layout()
        plt.show()
    print('Dependence plots complete.')


In [18]:
if RUN_SHAP:
    print('\nPlotting the selected-feature dependence panel...')
    target_features = ['EmployeeSalary', 'GovSub', 'HHI', 'DCG', 'TaxRate', 'AISkill', 'AnaAttention', 'REM', 'EmployeeTurnover']
    valid_features = [f for f in target_features if f in combined_X.columns]
    n_features = len(valid_features)
    if n_features == 0:
        print('None of the selected features is present in the data.')
    else:
        cols = 3
        rows = math.ceil(n_features / cols)
        fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows))
        if n_features > 1:
            axes = axes.flatten()
        else:
            axes = [axes]
        for i, feature_name in enumerate(valid_features):
            ax = axes[i]
            shap.dependence_plot(feature_name, combined_shap_values, combined_X, interaction_index=None, ax=ax, show=False, alpha=0.2, dot_size=15)
            x_data = combined_X[feature_name].values
            feature_idx = combined_X.columns.get_loc(feature_name)
            y_data = combined_shap_values[:, feature_idx]
            sort_idx = np.argsort(x_data)
            x_sorted = x_data[sort_idx]
            y_sorted = y_data[sort_idx]
            window_size = int(len(x_sorted) * 0.05)
            if window_size < 1:
                window_size = 1
            y_rolling = pd.Series(y_sorted).rolling(window=window_size, center=True).mean()
            ax.plot(x_sorted, y_rolling, color='black', linewidth=2.5, label='Mean Effect')
            ax.axhline(0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
            ax.set_title(f'{feature_name}', fontsize=13, fontweight='bold')
            ax.legend(loc='best')
        for j in range(i + 1, len(axes)):
            fig.delaxes(axes[j])
        plt.suptitle(f'SHAP Dependence Plots - {target_label}', fontsize=20, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.show()
    print('Selected-feature panel complete.')


## Part 8: Temporal analysis

Calculate annual normalized importance, examine its time trend and compare importance stability. The source screening uses p < 0.1 for temporal trends, and defines emerging features using an initial RVI below 1%; these are descriptive research screens, not confirmatory tests of causal effects.


In [19]:
if RUN_SHAP:
    print('\n' + '=' * 50)
    print('Analyzing feature importance over time...')
    print('=' * 50)
    years = [split[1] for split in train_test_splits]
    if len(years) != len(all_test_shap_values):
        print('The year and SHAP lists differ in length; check window alignment.')
    rvi_data = {}
    for i, year in enumerate(years):
        if i < len(all_test_shap_values):
            shap_vals = all_test_shap_values[i]
            mean_abs_shap = np.abs(shap_vals).mean(axis=0)
            total = mean_abs_shap.sum()
            if total == 0:
                total = 1
            rvi = mean_abs_shap / total
            rvi_data[year] = rvi
    rvi_df = pd.DataFrame(rvi_data, index=feature_cols)
    print('Plotting the annual feature-importance heatmap...')
    import matplotlib
    import matplotlib.pyplot as plt
    import seaborn as sns
    sorted_idx = rvi_df.mean(axis=1).sort_values(ascending=False).index
    rvi_df_sorted = rvi_df.loc[sorted_idx]
    sorted_rvi_matrix = rvi_df_sorted.values
    sorted_features = rvi_df_sorted.index.tolist()
    years_str = [str(y) for y in years]
    dynamic_height = max(10, len(sorted_features) * 0.25)
    fig, ax1 = plt.subplots(figsize=(12, dynamic_height))
    try:
        blues_enhanced = matplotlib.colormaps['Blues'].resampled(256).copy()
    except:
        blues_enhanced = matplotlib.cm.get_cmap('Blues').copy()
    blues_enhanced.set_gamma(0.5)
    vmin_percentile = np.percentile(sorted_rvi_matrix, 5)
    vmax_percentile = np.percentile(sorted_rvi_matrix, 95)
    im1 = ax1.imshow(sorted_rvi_matrix, aspect='auto', cmap=blues_enhanced, vmin=vmin_percentile, vmax=vmax_percentile)
    ax1.set_yticks(range(len(sorted_features)))
    ax1.set_yticklabels(sorted_features, fontsize=9)
    ax1.set_xticks(range(len(years)))
    ax1.set_xticklabels(years_str, fontsize=11, rotation=45)
    ax1.set_title(f'Time Variability of Variable Importance - {target_label}', fontsize=15, fontweight='bold', pad=20)
    ax1.set_xticks(np.arange(-0.5, len(years), 1), minor=True)
    ax1.set_yticks(np.arange(-0.5, len(sorted_features), 1), minor=True)
    ax1.grid(which='minor', color='gray', linestyle='-', linewidth=0.5, alpha=0.3)
    ax1.tick_params(which='minor', size=0)
    cbar1 = plt.colorbar(im1, ax=ax1, pad=0.02)
    cbar1.set_label('Normalized RVI Score', fontsize=12)
    plt.tight_layout()
    plt.show()
    trend_results = []
    X_time = np.arange(len(years))
    X_time_const = sm.add_constant(X_time)
    rank_df = rvi_df.rank(axis=0, ascending=False)
    for feature in rvi_df.index:
        y_rvi = rvi_df.loc[feature].values
        if np.var(y_rvi) < 1e-10:
            continue
        try:
            model = sm.OLS(y_rvi, X_time_const).fit()
            slope = model.params[1]
            p_value = model.pvalues[1]
            r_squared = model.rsquared
            start_rvi_fitted = model.params[0] + model.params[1] * X_time[0]
            end_rvi_fitted = model.params[0] + model.params[1] * X_time[-1]
            if start_rvi_fitted <= 1e-06:
                start_rvi_fitted = 1e-06
            change_pct = (end_rvi_fitted - start_rvi_fitted) / start_rvi_fitted
            avg_position = rank_df.loc[feature].mean()
            mean_rvi = y_rvi.mean()
            std_rvi = np.std(y_rvi)
            cv = std_rvi / mean_rvi if mean_rvi > 0 else 0
            trend_results.append({'Feature': feature, 'Slope': slope, 'P_Value': p_value, 'R2': r_squared, 'Change_Pct': change_pct, 'Start_RVI': start_rvi_fitted, 'End_RVI': end_rvi_fitted, 'Mean_RVI': mean_rvi, 'Std_RVI': std_rvi, 'CV': cv, 'Avg_Position': avg_position})
        except Exception as e:
            continue
    trend_df = pd.DataFrame(trend_results)
    top_10_important = trend_df.sort_values('Mean_RVI', ascending=False).head(10)
    core_stable_df = top_10_important.sort_values('CV', ascending=True).head(5)
    sig_change_df = trend_df[trend_df['P_Value'] < 0.1].copy()
    rising_df = sig_change_df[(sig_change_df['Slope'] > 0) & (sig_change_df['Start_RVI'] < 0.01)].sort_values('Slope', ascending=False)
    falling_df = sig_change_df[sig_change_df['Slope'] < 0].sort_values('Slope', ascending=True)
    print('\n' + '=' * 90)
    print('Temporal importance and stability report')
    print('=' * 90)
    print('\nStable features: the five lowest coefficients of variation among the top ten by mean importance')
    print(f"{'Rank':<4} {'Feature':<25} {'Mean RVI':<10} {'CV':<12} {'(mean rank)'}")
    print('-' * 75)
    for i, (idx, row) in enumerate(core_stable_df.iterrows(), 1):
        print(f"{i:<4} {row['Feature']:<25} {row['Mean_RVI']:<10.4f} {row['CV']:<12.4f} ({row['Avg_Position']:.1f})")
    print(f'\nFeatures with changing importance (p < 0.1):')
    print(f'  Total: {len(sig_change_df)} features')
    print(f'  Including {len(rising_df)} rising and {len(falling_df)} falling features')

    def print_trend_table(title, df):
        if df.empty:
            print(f'\n  {title}: none')
            return
        print(f'\n  {title}:')
        header = f"{'Rank':<4} {'Feature':<25} {'Slope':<12} {'p-value':<8} {'R²':<8} {'Start RVI':<10} {'End RVI':<10}"
        print(header)
        print('-' * 90)
        for i, (idx, row) in enumerate(df.iterrows(), 1):
            print(f"{i:<4} {row['Feature']:<25} {row['Slope']:>+10.6f}    {row['P_Value']:.3f}    {row['R2']:.3f}     {row['Start_RVI']:.4f}     {row['End_RVI']:.4f}")
    print_trend_table('Emerging features (initial RVI < 1%, ranked by slope)', rising_df)
    print_trend_table('Declining features (ranked by slope)', falling_df)
    print('-' * 90)
    print('Plotting the five leading emerging features...')
    top_rising_plot = rising_df.head(5)
    if not top_rising_plot.empty:
        plt.figure(figsize=(12, 7))
        import matplotlib.cm as cm
        cmap = plt.get_cmap('tab10')
        for i, (idx, row) in enumerate(top_rising_plot.iterrows()):
            feat = row['Feature']
            color = cmap(i % cmap.N)
            plt.plot(years, rvi_df.loc[feat], marker='o', linestyle='-', linewidth=2.5, color=color, label=feat)
        plt.title(f'Temporal Evolution of Top 5 Rising Features - {target_label}', fontsize=15, fontweight='bold')
        plt.xlabel('Year', fontsize=12)
        plt.ylabel('RVI Score (Relative Importance)', fontsize=12)
        plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0.0, fontsize=10, title='Top 5 Rising Features')
        plt.grid(True, linestyle='--', alpha=0.4)
        plt.tight_layout()
        plt.show()
    else:
        print('No qualifying emerging features; plot skipped.')


## Part 9: Final-window interaction analysis

The supplied interaction block uses the final fitted GBRT model over the pooled feature rows. This scope differs from the window-specific out-of-sample SHAP values used for the RVI analysis above. It is kept as a separate, optional illustration rather than presented as an additional held-out performance evaluation.


In [20]:
if RUN_SHAP and RUN_INTERACTIONS:
    print('\n' + '=' * 80)
    print('Calculating SHAP interactions for selected features...')
    print('=' * 80)
    target_interact_vars = ['EmployeeSalary', 'GovSub', 'HHI', 'DCG', 'TaxRate', 'AISkill', 'AnaAttention', 'REM', 'EmployeeTurnover']
    valid_targets = [var for var in target_interact_vars if var in combined_X.columns]
    if set(target_interact_vars) - set(valid_targets):
        print(f'Skipping unavailable features: {set(target_interact_vars) - set(valid_targets)}')
    if valid_targets:
        explainer_interact = shap.TreeExplainer(best_model)
        print('Calculating the interaction matrix...')
        shap_interactions = explainer_interact.shap_interaction_values(combined_X)
        mean_abs_interaction = np.abs(shap_interactions).mean(axis=0)
        features = combined_X.columns.tolist()
        feature_total_importance = np.abs(combined_shap_values).mean(axis=0)
        global_total_importance = feature_total_importance.sum()
        for target_var in valid_targets:
            print(f'\n' + '-' * 80)
            print(f'Selected feature: [{target_var}】')
            print('-' * 80)
            target_idx = features.index(target_var)
            target_total_shap = feature_total_importance[target_idx]
            interaction_records = []
            for j, feature_j in enumerate(features):
                if j == target_idx:
                    continue
                interaction_score = mean_abs_interaction[target_idx, j] * 2
                pct_of_var1 = mean_abs_interaction[target_idx, j] / target_total_shap * 100
                pct_of_global = interaction_score / global_total_importance * 100
                interaction_records.append({'Variable1': target_var, 'Variable2': feature_j, 'Score': interaction_score, 'Pct_of_Var1': pct_of_var1, 'Pct_of_Global': pct_of_global})
            interaction_df = pd.DataFrame(interaction_records)
            interaction_df = interaction_df.sort_values(by='Score', ascending=False).reset_index(drop=True)
            top_10 = interaction_df.head(10)
            top_10.index = np.arange(1, len(top_10) + 1)
            print(f'\n[{target_var}] top ten interaction pairs:')
            print(f"{'Rank':<4} | {'Variable1':<18} | {'Variable2':<18} | {'Score':<8} | {'% of Var1':<10} | {'% of Global':<10}")
            print('-' * 82)
            for rank, row in top_10.iterrows():
                print(f"{rank:<4} | {row['Variable1']:<18} | {row['Variable2']:<18} | {row['Score']:<8.4f} | {row['Pct_of_Var1']:>7.2f}% | {row['Pct_of_Global']:>8.2f}%")
            top_v2 = top_10.iloc[0]['Variable2']
            pct_v1_display = top_10.iloc[0]['Pct_of_Var1']
            print(f'\nLeading interaction: {top_v2} (share of {target_var} feature importance: {pct_v1_display:.2f}%)')
            print(f'Plotting [{target_var}] & [{top_v2}] interaction dependence...')
            fig, ax = plt.subplots(figsize=(8, 6))
            shap.dependence_plot(target_var, combined_shap_values, combined_X, interaction_index=top_v2, ax=ax, show=False, alpha=0.6, dot_size=15, cmap=plt.get_cmap('coolwarm'))
            ax.set_title(f'Interaction: {target_var} & {top_v2}', fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.show()
    print('\nSelected-feature interaction analysis complete.')


## Part 10: Research summary

The final thesis finds that GBRT predicts both innovation targets more accurately than the RF and OLS comparators. Firm fundamentals and government-related variables are prominent across targets. Employee and board characteristics are particularly informative for R&D inputs, while management, stock-market and supply-chain features are more prominent for invention patent output.

SHAP analysis organizes the predictive information into feature rankings, group contributions, dependence patterns, interactions and temporal changes. These findings inform research and resource-allocation priorities without establishing causal effects.

See the [final thesis](../paper/corporate_innovation_thesis.pdf) and [research overview](../docs/research-overview.md) for the complete reported results and presentation figures.
